<a href="https://colab.research.google.com/github/a01669283/Equipo2-Cero-Fraude-Aplicacion/blob/main/Integrated_Activity_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Integrated Activity 1
# Authors: Armando Vásquez Ambrocio - A01669283, Hugo Fernando Rodríguez Montiel - A01782589
# Date: September 25, 2026

# **Part 1 - Malicious code inside transmissions**

Reviewing if the content of `mcodeX.txt` appears inside `transmission1.txt` and `transmission2.txt`.

The possible outcomes are the following:
- If found -> `true` + the starting position of the match in the transmission
- If not found -> `false`

In [ ]:
import os

# Reads one input file and returns its content without surrounding whitespace.
# Parameters:
#   fileName: name of the file, expected in the same folder as the notebook
# Returns:
#   string with the content of the file

def readInputFile(fileName: str) -> str:
  filePath = os.path.join(os.getcwd(), fileName)

  # Validation: stop with a clear message if the file was not uploaded
  if not os.path.exists(filePath):
    raise FileNotFoundError(f"Missing input file: {fileName}")

  with open(filePath, "r", encoding="utf-8") as inputFile:
    return inputFile.read().strip()

In [ ]:
# Collects basic information about a loaded input file.
# Parameters:
#   fileName: name of the file, used only for the report
#   content: text returned by readInputFile
# Returns:
#   dictionary with length, alphabet and preview of the content

def describeInputFile(fileName: str, content: str) -> dict:
  alphabet = sorted(set(content))

  return {
      "file": fileName,
      "length": len(content),
      "alphabetSize": len(alphabet),
      "alphabet": "".join(alphabet),
      "preview": content[:20]
  }

In [ ]:
# Loads every input file and prints a summary table.
# Parameters:
#   fileNames: list with the names of the files to analyze
# Returns:
#   dictionary that maps each file name to its content

def analyzeInputFiles(fileNames: list[str]) -> dict:
  contents = {}

  print(f"{'FILE':<20}{'LENGTH':>12}{'ALPHABET SIZE':>10}  PREVIEW")

  for fileName in fileNames:
    content = readInputFile(fileName)
    contents[fileName] = content
    report = describeInputFile(fileName, content)

    print(f"{report['file']:<20}{report['length']:>12}{report['alphabetSize']:>10}  "
          f"{report['preview']}")
    print(f"{'':<20}characters used: {report['alphabet']}")

  return contents

In [ ]:
# Builds the KMP prefix table: for each position, the length of the longest beginning of the pattern that is also an ending at that position.
# Parameters:
#   pattern: string to look for
# Returns:
#   list of ints with the prefix table (same size as pattern)

def findPrefix(pattern: str) -> list[int]:
  patternLength = len(pattern)
  prefixArray = [0] * patternLength
  prefixLength = 0

  for i in range(1, patternLength):
    while prefixLength > 0 and pattern[i] != pattern[prefixLength]:
      prefixLength = prefixArray[prefixLength - 1]

    if pattern[i] == pattern[prefixLength]:
      prefixLength = prefixLength + 1

    prefixArray[i] = prefixLength

  return prefixArray

In [ ]:
# Finds the first occurrence of a pattern inside a text using KMP.
# Parameters:
#   text: string where the pattern is searched
#   pattern: string to look for
# Returns:
#   0-indexed start of the first match, or -1 if there is none

def kmpFirstMatch(text: str, pattern: str) -> int:
  textLength = len(text)
  patternLength = len(pattern)

  # Validation: An empty pattern, or one longer than the text, can never match
  if patternLength == 0 or patternLength > textLength:
    return -1

  prefixArray = findPrefix(pattern)

  j = 0
  for i in range(textLength):
    while j > 0 and text[i] != pattern[j]:
      j = prefixArray[j - 1]

    if text[i] == pattern[j]:
      j = j + 1

    if j == patternLength:
      return i - patternLength + 1
  return -1

In [ ]:
# Uploads the 5 test files into the session folder.

from google.colab import files

# Removes old copies for testing
for fileName in ["transmission1.txt", "transmission2.txt",
                 "mcode1.txt", "mcode2.txt", "mcode3.txt"]:
  if os.path.exists(fileName):
    os.remove(fileName)

uploadedFiles = files.upload()
print(f"Files uploaded: {len(uploadedFiles)}")

Saving mcode1.txt to mcode1.txt
Saving mcode2.txt to mcode2.txt
Saving mcode3.txt to mcode3.txt
Saving transmission1.txt to transmission1.txt
Saving transmission2.txt to transmission2.txt
Files uploaded: 5


In [ ]:
# Loads the input files used by Parts 1, 2 and 3.

INPUT_FILES = ["transmission1.txt", "transmission2.txt",
               "mcode1.txt", "mcode2.txt", "mcode3.txt"]

contents = analyzeInputFiles(INPUT_FILES)

In [ ]:
# Part 1: checks every mcode inside every transmission and prints the result.

TRANSMISSION_FILES = ["transmission1.txt", "transmission2.txt"]
MCODE_FILES = ["mcode1.txt", "mcode2.txt", "mcode3.txt"]

for transmissionName in TRANSMISSION_FILES:
  for mcodeName in MCODE_FILES:
    position = kmpFirstMatch(contents[transmissionName], contents[mcodeName])

    if position == -1:
      result = "false"
    else:
      result = f"true {position + 1}"

    print(f"{transmissionName} - {mcodeName}: {result}")

# **Part 2 - Mirrored code inside transmissions**

Finding the longest palindrome (a sequence of characters that reads the same forwards and backwards) inside `transmission1.txt` and `transmission2.txt`.

The outcome for each transmission is the following:
- The starting and ending positions (1-indexed) of the longest palindrome found

In [ ]:
# Expands outward from a center while both sides match.
# Parameters:
#   text: string where the palindrome is searched
#   left: starting index of the left side of the center
#   right: starting index of the right side of the center
# Returns:
#   tuple (start, end) with the 0-indexed limits of the palindrome found;
#   if no palindrome is found, start is greater than end

def expandFromCenter(text: str, left: int, right: int) -> tuple[int, int]:
  textLength = len(text)

  while left >= 0 and right < textLength and text[left] == text[right]:
    left = left - 1
    right = right + 1

  # The loop stops one step outside the palindrome, so step back one position on each side
  return left + 1, right - 1

In [ ]:
# Finds the longest palindrome inside a text using expand-around-center.
# Parameters:
#   text: string where the palindrome is searched
# Returns:
#   tuple (start, end) with the 1-indexed limits of the first longest palindrome,
#   or (0, 0) if the text is empty

def longestPalindrome(text: str) -> tuple[int, int]:
  # Validation: an empty text has no palindrome
  if len(text) == 0:
    return 0, 0

  maxPalindromeStart = 0
  maxPalindromeEnd = 0

  for center in range(len(text)):
    # Odd length: the middle is one character
    oddStart, oddEnd = expandFromCenter(text, center, center)
    # Even length: the middle is the gap between two characters
    evenStart, evenEnd = expandFromCenter(text, center, center + 1)

    # Strict ">" keeps the first palindrome when there is a tie
    if oddEnd - oddStart > maxPalindromeEnd - maxPalindromeStart:
      maxPalindromeStart, maxPalindromeEnd = oddStart, oddEnd

    if evenEnd - evenStart > maxPalindromeEnd - maxPalindromeStart:
      maxPalindromeStart, maxPalindromeEnd = evenStart, evenEnd

  return maxPalindromeStart + 1, maxPalindromeEnd + 1

In [ ]:
# Part 2: finds the longest palindrome inside every transmission and prints the limit values.

for transmissionName in TRANSMISSION_FILES:
  start, end = longestPalindrome(contents[transmissionName])
  print(f"{transmissionName}: {start} {end}")

transmission1.txt: 1816193 1816203
transmission2.txt: 1816121 1816131


# **Part 3 - Common code between transmissions**

Finding the longest common substring (a sequence of consecutive characters that appears in both files) between `transmission1.txt` and `transmission2.txt`.

The outcome is the following:
- The starting and ending positions of the longest common substring
- The starting and ending positions (1-indexed) of the longest common substring

In [ ]:
# Constants for the rolling hash: a base bigger than any byte value and a large prime modulus
BASE = 257
MOD = (1 << 61) - 1

# Computes the hash of every window of a given length, sliding one character at a time.
# Parameters:
#   data: bytes of the text
#   length: size of the window
# Yields:
#   tuple (start, hash) for every window; start is 0-indexed

def slidingHashes(data: bytes, length: int):
  windowHash = 0
  for i in range(length):
    windowHash = (windowHash * BASE + data[i]) % MOD
  yield 0, windowHash

  # Weight of the character that leaves the window
  highestPower = pow(BASE, length - 1, MOD)

  for i in range(length, len(data)):
    windowHash = ((windowHash - data[i - length] * highestPower) * BASE + data[i]) % MOD
    yield i - length + 1, windowHash

In [ ]:
# Checks if both texts share a substring of a given length.
# Parameters:
#   text1, text2: the texts to compare
#   data1, data2: the same texts as bytes
#   length: size of the substring to look for
# Returns:
#   0-indexed start in text1 of the first common substring, or -1 if there is none

def findCommonOfLength(text1: str, text2: str, data1: bytes, data2: bytes, length: int) -> int:
  hashesText2 = set(windowHash for _, windowHash in slidingHashes(data2, length))

  for start, windowHash in slidingHashes(data1, length):
    if windowHash in hashesText2:
      # Validation: confirm the match with the real characters (hash collisions are possible)
      if text1[start:start + length] in text2:
        return start

  return -1

In [ ]:
# Finds the longest common substring of two texts using binary search on its length.
# Parameters:
#   text1: first text (positions are reported in this one)
#   text2: second text
# Returns:
#   tuple (start, end) with the 1-indexed limits in text1, or (0, 0) if there is none

def longestCommonSubstring(text1: str, text2: str) -> tuple[int, int]:
  data1 = text1.encode()
  data2 = text2.encode()

  low = 1
  high = min(len(text1), len(text2))
  bestStart = -1
  bestLength = 0

  while low <= high:
    middle = (low + high) // 2
    start = findCommonOfLength(text1, text2, data1, data2, middle)

    if start != -1:
      bestStart, bestLength = start, middle
      low = middle + 1
    else:
      high = middle - 1

  # Validation: the texts share no character
  if bestLength == 0:
    return 0, 0

  return bestStart + 1, bestStart + bestLength

In [ ]:
# Part 3: finds the longest common substring between both transmissions and prints its limits.

start, end = longestCommonSubstring(contents["transmission1.txt"], contents["transmission2.txt"])
print(f"transmission1.txt - transmission2.txt: {start} {end}")